# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [ ]:
!pip install transformers torch scikit-learn pandas accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
import itertools
import os
import urllib.request
import zipfile
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import (
    BertTokenizer,
    pipeline,
    BertModel,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
# 1. 事前学習済みBERTモデルのトークナイザーをロード
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# 2. 対象のテキスト
text = "The movie was full of incomprehensibilities."

# 3. テキストをトークン化 (WordPieceアルゴリズム)
tokens = tokenizer.tokenize(text)

# 4. 結果の表示
print("元のテキスト:", text)
print("トークン列  :", tokens)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

元のテキスト: The movie was full of incomprehensibilities.
トークン列  : ['the', 'movie', 'was', 'full', 'of', 'inc', '##omp', '##re', '##hen', '##si', '##bilities', '.']


## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [ ]:
# 1. 穴埋めタスク（fill-mask）用のパイプラインをロード
unmasker = pipeline('fill-mask', model='bert-base-uncased')

# 2. 対象のテキスト
text = "The movie was full of [MASK]."

# 3. 予測の実行（デフォルトで上位5つの候補を返す）
results = unmasker(text)

# 4. 結果の表示
print(f"入力テキスト: {text}\n")
print("--- 予測された上位のトークン ---")

for i, res in enumerate(results):
    token = res['token_str']
    score = res['score']
    sequence = res['sequence']
    print(f"Top {i+1}: '{token}' (確率: {score:.4f})")
    print(f"         完成した文: {sequence}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


入力テキスト: The movie was full of [MASK].

--- 予測された上位のトークン ---
Top 1: 'fun' (確率: 0.1071)
         完成した文: the movie was full of fun.
Top 2: 'surprises' (確率: 0.0663)
         完成した文: the movie was full of surprises.
Top 3: 'drama' (確率: 0.0447)
         完成した文: the movie was full of drama.
Top 4: 'stars' (確率: 0.0272)
         完成した文: the movie was full of stars.
Top 5: 'laughs' (確率: 0.0254)
         完成した文: the movie was full of laughs.


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [ ]:
# 3. 予測の実行（デフォルトで上位5つの候補を返します）
results = unmasker(text, top_k=10)

# 4. 結果の表示
print(f"入力テキスト: {text}\n")
print("--- 予測された上位のトークン ---")

# 確率（スコア）が高い順に結果を出力
for i, res in enumerate(results):
    token = res['token_str']
    score = res['score']
    sequence = res['sequence']
    print(f"Top {i+1}: '{token}' (確率: {score:.4f})")
    print(f"         完成した文: {sequence}")

入力テキスト: The movie was full of [MASK].

--- 予測された上位のトークン ---
Top 1: 'fun' (確率: 0.1071)
         完成した文: the movie was full of fun.
Top 2: 'surprises' (確率: 0.0663)
         完成した文: the movie was full of surprises.
Top 3: 'drama' (確率: 0.0447)
         完成した文: the movie was full of drama.
Top 4: 'stars' (確率: 0.0272)
         完成した文: the movie was full of stars.
Top 5: 'laughs' (確率: 0.0254)
         完成した文: the movie was full of laughs.
Top 6: 'action' (確率: 0.0195)
         完成した文: the movie was full of action.
Top 7: 'excitement' (確率: 0.0190)
         完成した文: the movie was full of excitement.
Top 8: 'people' (確率: 0.0183)
         完成した文: the movie was full of people.
Top 9: 'tension' (確率: 0.0150)
         完成した文: the movie was full of tension.
Top 10: 'music' (確率: 0.0146)
         完成した文: the movie was full of music.


## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [ ]:
# 1. モデルとトークナイザーの準備
model = BertModel.from_pretrained('bert-base-uncased')
model.eval() # 評価モードに設定

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
# 2. 対象となる文のリスト
sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
    ]

# 各文のベクトルを計算して保存
print("文のベクトル化を計算中...\n")

# トークン化してPyTorchのテンソル(pt)に変換
inputs = tokenizer(sentences, return_tensors="pt")

# 勾配計算を無効化して推論を実行
with torch.no_grad():
    # 辞書のキーを引数名に、値をその中身にして全部まとめて渡す
    outputs = model(**inputs)

# 最終層の隠れ状態 (バッチサイズ, トークン長, 隠れ層の次元数)
last_hidden_state = outputs.last_hidden_state

# 先頭のトークンが [CLS] トークン
# すべての文の[CLS]トークンのベクトルを取り出す->len(cls_embedding) = 4
cls_embedding = last_hidden_state[:, 0, :]

# それぞれのベクトルを辞書型（key=テキスト）として格納
embeddings = {sentence: vector for sentence, vector in zip(sentences, cls_embedding)}

# 4. 全組み合わせ（ペア）の生成とコサイン類似度の計算
print("--- コサイン類似度 (Cosine Similarity) ---")

# itertools.combinations を使って、順序を問わない全ペア(4C2 = 6通り)を生成
pairs = list(itertools.combinations(sentences, 2))

# 類似度を計算する
results = []
for s1, s2 in pairs:
    emb1 = embeddings[s1]
    emb2 = embeddings[s2]

    # コサイン類似度を計算 (-1.0 〜 1.0 の値を取る)
    # emb1, emb2はどちらも1次元なので、dim=0とする
    sim = F.cosine_similarity(emb1, emb2, dim=0).item()
    results.append((sim, s1, s2))

for sim, s1, s2 in results:
    print(f"類似度: {sim:.4f}")
    print(f" A: {s1}")
    print(f" B: {s2}")
    print("-" * 40)

文のベクトル化を計算中...

--- コサイン類似度 (Cosine Similarity) ---
類似度: 0.9881
 A: The movie was full of fun.
 B: The movie was full of excitement.
----------------------------------------
類似度: 0.9558
 A: The movie was full of fun.
 B: The movie was full of crap.
----------------------------------------
類似度: 0.9475
 A: The movie was full of fun.
 B: The movie was full of rubbish.
----------------------------------------
類似度: 0.9541
 A: The movie was full of excitement.
 B: The movie was full of crap.
----------------------------------------
類似度: 0.9487
 A: The movie was full of excitement.
 B: The movie was full of rubbish.
----------------------------------------
類似度: 0.9807
 A: The movie was full of crap.
 B: The movie was full of rubbish.
----------------------------------------


## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

In [ ]:
# 2. 対象となる文のリスト
sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
    ]

# 各文のベクトルを計算して保存
print("文のベクトル化を計算中...\n")

# トークン化してPyTorchのテンソル(pt)に変換
inputs = tokenizer(sentences, return_tensors="pt")

# 勾配計算を無効化して推論を実行
with torch.no_grad():
    # 辞書のキーを引数名に、値をその中身にして全部まとめて渡す
    outputs = model(**inputs)

# 最終層の隠れ状態 (バッチサイズ, トークン長, 隠れ層の次元数)
last_hidden_state = outputs.last_hidden_state

# 最終層のすべてのトークンのベクトルを平均する
# dim=1 が「トークン長」の次元なので、ここで平均をとる
mean_embeddings = last_hidden_state.mean(dim=1)

# それぞれのベクトルを辞書型（key=テキスト）として格納
embeddings = {sentence: vector for sentence, vector in zip(sentences, mean_embeddings)}

# 4. 全組み合わせ（ペア）の生成とコサイン類似度の計算
print("--- コサイン類似度 (Cosine Similarity) ---")

# itertools.combinations を使って、順序を問わない全ペア(4C2 = 6通り)を生成
pairs = list(itertools.combinations(sentences, 2))

# 類似度を計算する
results = []
for s1, s2 in pairs:
    emb1 = embeddings[s1]
    emb2 = embeddings[s2]

    # コサイン類似度を計算 (-1.0 〜 1.0 の値を取る)
    # emb1, emb2はどちらも1次元なので、dim=0とする
    sim = F.cosine_similarity(emb1, emb2, dim=0).item()
    results.append((sim, s1, s2))

for sim, s1, s2 in results:
    print(f"類似度: {sim:.4f}")
    print(f" A: {s1}")
    print(f" B: {s2}")
    print("-" * 40)

文のベクトル化を計算中...

--- コサイン類似度 (Cosine Similarity) ---
類似度: 0.9568
 A: The movie was full of fun.
 B: The movie was full of excitement.
----------------------------------------
類似度: 0.8490
 A: The movie was full of fun.
 B: The movie was full of crap.
----------------------------------------
類似度: 0.8169
 A: The movie was full of fun.
 B: The movie was full of rubbish.
----------------------------------------
類似度: 0.8352
 A: The movie was full of excitement.
 B: The movie was full of crap.
----------------------------------------
類似度: 0.7938
 A: The movie was full of excitement.
 B: The movie was full of rubbish.
----------------------------------------
類似度: 0.9226
 A: The movie was full of crap.
 B: The movie was full of rubbish.
----------------------------------------


## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

In [ ]:
# 1. SST-2データセットをダウンロードして解凍する

base_dir="./data"
url = "https://dl.fbaipublicfiles.com/glue/data/SST-2.zip"
zip_path = os.path.join(base_dir, "SST-2.zip")
sst2_dir = os.path.join(base_dir, "SST-2")

# 保存先ディレクトリの作成
os.makedirs(base_dir, exist_ok=True)

print("SST-2データセットをダウンロードしています...")
urllib.request.urlretrieve(url, zip_path)

print("ダウンロード完了。ZIPファイルを解凍しています...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(base_dir)
print("解凍完了。\n")

SST-2データセットをダウンロードしています...
ダウンロード完了。ZIPファイルを解凍しています...
解凍完了。



In [ ]:
# 2. ファイルパスの指定
train_path = os.path.join(sst2_dir, "train.tsv")  # 学習用データ
dev_path = os.path.join(sst2_dir, "dev.tsv")      # 評価用データ

# 3. pandasでTSVファイルを読み込み
# SST-2は 'sentence'（テキスト）と 'label'（極性: 0=ネガティブ, 1=ポジティブ）の列を持ちます
print("TSVファイルを読み込んでいます...")
train_df = pd.read_csv(train_path, sep='\t')
dev_df = pd.read_csv(dev_path, sep='\t')

# 5. トークン化の実行
# 各データフレームに新しく 'tokens' という列を作成し、トークン化されたリストを格納します
train_df['tokens'] = [tokenizer.tokenize(str(text)) for text in train_df['sentence']]
dev_df['tokens'] = [tokenizer.tokenize(str(text)) for text in dev_df['sentence']]

print("処理が完了しました。\n")

# 結果の確認（Trainセットの最初の3件を表示）
print("--- 訓練セット (Train) の先頭3件 ---")
for index, row in train_df.head(3).iterrows():
    print(f"Index : {index}")
    print(f"Label : {row['label']} (0:Negative, 1:Positive)")
    print(f"Text  : {row['sentence']}")
    print(f"Tokens: {row['tokens']}")
    print("-" * 50)

TSVファイルを読み込んでいます...
処理が完了しました。

--- 訓練セット (Train) の先頭3件 ---
Index : 0
Label : 0 (0:Negative, 1:Positive)
Text  : hide new secretions from the parental units 
Tokens: ['hide', 'new', 'secret', '##ions', 'from', 'the', 'parental', 'units']
--------------------------------------------------
Index : 1
Label : 0 (0:Negative, 1:Positive)
Text  : contains no wit , only labored gags 
Tokens: ['contains', 'no', 'wit', ',', 'only', 'labor', '##ed', 'gag', '##s']
--------------------------------------------------
Index : 2
Label : 1 (0:Negative, 1:Positive)
Text  : that loves its characters and communicates something rather beautiful about human nature 
Tokens: ['that', 'loves', 'its', 'characters', 'and', 'communicate', '##s', 'something', 'rather', 'beautiful', 'about', 'human', 'nature']
--------------------------------------------------


## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

In [ ]:
data = train_df[:4]

train_df = pd.DataFrame(data)


# 2. ミニバッチ化するテキストとラベルのリストを取得
batch_sentences = train_df['sentence'].tolist()
batch_labels = train_df['label'].tolist()

# 3. トークナイザーで一括処理（パディングとテンソル化）
# padding=True : バッチ内の最大長に合わせて [PAD] トークンを追加
# truncation=True : モデルの最大入力長を超える場合は切り捨てる
# return_tensors="pt" : PyTorchのテンソル形式で出力する
batch_inputs = tokenizer(
    batch_sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# ラベルもPyTorchのテンソルに変換
labels_tensor = torch.tensor(batch_labels)

In [ ]:
# 4. 結果の確認
print("=== ミニバッチのテンソル形状 ===")
print(f"Input IDs shape     : {batch_inputs['input_ids'].shape} -> (バッチサイズ, トークン長)")
print(f"Attention Mask shape: {batch_inputs['attention_mask'].shape}")
print(f"Labels shape        : {labels_tensor.shape}\n")

print("=== Input IDs (トークンID) ===")
print(batch_inputs['input_ids'])
print("※ ID=0 が [PAD] トークン、文頭(101=[CLS])、文末(102=[SEP])\n")

print("=== Attention Mask ===")
print(batch_inputs['attention_mask'])
print("※ 1は有効なトークン、0は [PAD] トークン\n")

print("=== トークン列の視覚的な確認 ===")
for i, ids in enumerate(batch_inputs['input_ids']):
    # IDを再び文字列のトークンに変換して表示
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"文{i}: {tokens}")

=== ミニバッチのテンソル形状 ===
Input IDs shape     : torch.Size([4, 15]) -> (バッチサイズ, トークン長)
Attention Mask shape: torch.Size([4, 15])
Labels shape        : torch.Size([4])

=== Input IDs (トークンID) ===
tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
             0,     0,     0,     0,     0],
        [  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102,     0,     0,     0,     0],
        [  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          3376,  2055,  2529,  3267,   102],
        [  101,  3464, 12580,  8510,  2000,  3961,  1996,  2168,  2802,   102,
             0,     0,     0,     0,     0]])
※ ID=0 が [PAD] トークン、文頭(101=[CLS])、文末(102=[SEP])

=== Attention Mask ===
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])
※ 1は有効なトークン、0は [PAD]

## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [ ]:
# ==========================================
# 1. PyTorch用のカスタムデータセットクラス
# ==========================================
class SST2Dataset(Dataset):
    def __init__(self, encodings, labels):
        # トークン化済みのテキストをSST2Dataset.encodingsに、正解ラベルをSST2Dataset.labelsに格納
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # 指定されたインデックスのデータをテンソル形式で返す
        # self.encodingsはすべてのデータが格納されている
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        # 全部で何行データがあるのか
        return len(self.labels)

# ==========================================
# 2. 評価指標（正解率）を計算する関数
# ==========================================
def compute_metrics(pred):
    # 正解のデータ
    labels = pred.label_ids

    # 予測された確率の中で最も高いクラス（0または1）を取得
    preds = pred.predictions.argmax(-1)

    # scikit-learnの関数で正解率を計算
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
    }

# ==========================================
# 3. メインの学習・評価処理
# ==========================================

# --- トークン化 ---
# paddingとtruncationを有効にして一括処理
train_encodings = tokenizer(train_df['sentence'].tolist(), truncation=True, padding=True, max_length=128)
dev_encodings = tokenizer(dev_df['sentence'].tolist(), truncation=True, padding=True, max_length=128)

# PyTorchのDatasetオブジェクトに変換
train_dataset = SST2Dataset(train_encodings, train_df['label'].tolist())
dev_dataset = SST2Dataset(dev_encodings, dev_df['label'].tolist())

# --- モデルのロード ---
print("分類用のBERTモデルをロードしています...")
# num_labels=2 (ネガティブ/ポジティブの2値分類) を指定します
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# --- トレーニング設定 ---
training_args = TrainingArguments(
    output_dir='./results',          # モデルの保存先
    num_train_epochs=3,              # エポック数（データセットを何周学習させるか）
    per_device_train_batch_size=16,  # 訓練時のバッチサイズ
    per_device_eval_batch_size=32,   # 評価時のバッチサイズ
    warmup_steps=500,                # 学習率のウォームアップステップ数
    weight_decay=0.01,               # 過学習を防ぐための重み減衰
    logging_dir='./logs',            # ログの保存先
    logging_steps=1,                 # 100ステップごとにログを出力
    eval_strategy="epoch",           # エポックごとに評価を実行
    save_strategy="epoch",           # エポックごとにモデルを保存
    load_best_model_at_end=True,     # 最後に最も性能の良かったモデルをロード
)

# --- Trainerの初期化 ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics, # 評価関数を指定
)

# --- ファインチューニングの実行 ---
print("学習を開始します...")
trainer.train()

# --- 検証セット(dev)での最終評価 ---
print("\n学習が完了しました。検証セットでの最終評価を行います...")
eval_results = trainer.evaluate()

print("\n=== 最終評価結果 (Validation Set) ===")
print(f"Loss (損失) : {eval_results['eval_loss']:.4f}")
print(f"Accuracy (正解率) : {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")

分類用のBERTモデルをロードしています...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

学習を開始します...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.580109,0.731108,0.490826
2,0.630190,0.731103,0.490826
3,0.615953,0.731111,0.490826


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


学習が完了しました。検証セットでの最終評価を行います...



=== 最終評価結果 (Validation Set) ===
Loss (損失) : 0.7311
Accuracy (正解率) : 0.4908 (49.08%)


## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [ ]:
device = next(model.parameters()).device

model.eval() # 推論モードに設定

# 2. 対象となる文のリスト
sentences = [
    "The movie was full of incomprehensibilities.",
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# ラベルのマッピング
label_map = {0: "Negative", 1: "Positive"}

print("=== 極性予測の実行 ===\n")

# 3. 各文に対して予測を実行
for text in sentences:
    # トークン化してPyTorchテンソルに変換
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    inputs = {key: val.to(device) for key, val in inputs.items()}

    # 勾配計算を無効化して推論
    with torch.no_grad():
        outputs = model(**inputs)

    # モデルの出力を取得
    logits = outputs.logits

    # ソフトマックス関数を適用して確率に変換
    probabilities = F.softmax(logits, dim=1).squeeze()

    # 最も確率が高いクラスのインデックス（整数）を取得
    predicted_class_id = torch.argmax(probabilities).item()
    # 番号をラベル名に変換
    predicted_label = label_map[predicted_class_id]
    # 確信度を取得する
    confidence = probabilities[predicted_class_id].item()

    # 結果の表示
    print(f"テキスト: {text}")
    print(f"予測結果: {predicted_label} (確信度: {confidence:.4f})")
    print("-" * 50)

=== 極性予測の実行 ===

テキスト: The movie was full of incomprehensibilities.
予測結果: Positive (ポジティブ) (確信度: 0.5124)
--------------------------------------------------
テキスト: The movie was full of fun.
予測結果: Negative (ネガティブ) (確信度: 0.5024)
--------------------------------------------------
テキスト: The movie was full of excitement.
予測結果: Negative (ネガティブ) (確信度: 0.5048)
--------------------------------------------------
テキスト: The movie was full of crap.
予測結果: Positive (ポジティブ) (確信度: 0.5317)
--------------------------------------------------
テキスト: The movie was full of rubbish.
予測結果: Positive (ポジティブ) (確信度: 0.5068)
--------------------------------------------------


## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [ ]:
# ==========================================
# 1. カスタムモデルの定義 (Max Pooling Architecture)
# ==========================================
# PyTorchの nn.Module を自分の新しいクラスに引き継ぐ
class CustomBertMaxPoolingClassifier(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_labels=2):
        super().__init__()

        # 分類用のヘッドがついていないBertModelをロード（隠れ状態を直接取り出せる）
        self.bert = BertModel.from_pretrained(model_name)

        self.num_labels = num_labels

        # 分類用の全結合層
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        # 1. BERTモデルを通過させ、最終層の隠れ状態を取得
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # last_hidden_state の形状: (バッチサイズ, トークン長, 隠れ層の次元数)
        last_hidden_state = outputs.last_hidden_state

        # 2. Max Pooling の計算
        # attention_mask を考慮し、[PAD] トークンが最大値として選ばれないように極端な負の値(-1e9)に置き換える
        mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size())

        # マスクが1（有効なトークン）ならそのまま、0（パディング）なら -1e9 にする（パディングが選ばれないようにする）
        masked_hidden_state = torch.where(
            mask_expanded == 1,
            last_hidden_state,
            torch.tensor(-1e9).to(last_hidden_state.device)
        )

        # トークン長（dim=1）の次元で最大値を取得 -> 形状: (バッチサイズ, 隠れ層の次元数)
        max_pooled_output = torch.max(masked_hidden_state, dim=1).values

        # 3. 分類層（Linear）に入力
        logits = self.classifier(max_pooled_output)

        # 4. 損失の計算（学習用）
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        # Trainer API が解釈できるように SequenceClassifierOutput 形式で返す
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

# ==========================================
# 3. ファインチューニングの実行
# ==========================================

print("テキストをトークン化しています...")
train_encodings = tokenizer(train_df['sentence'].tolist(), truncation=True, padding=True, max_length=128)
dev_encodings = tokenizer(dev_df['sentence'].tolist(), truncation=True, padding=True, max_length=128)

train_dataset = SST2Dataset(train_encodings, train_df['label'].tolist())
dev_dataset = SST2Dataset(dev_encodings, dev_df['label'].tolist())

print("カスタムモデル (Max Pooling Architecture) を初期化しています...")
model = CustomBertMaxPoolingClassifier(model_name='bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results_custom',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_custom',
    logging_steps=1,
    eval_strategy="epoch",  # エラー回避のため新しい引数名を使用
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

print("学習を開始します...")
trainer.train()

print("\n=== カスタムモデルによる最終評価 (Validation Set) ===")
eval_results = trainer.evaluate()
print(f"Loss (損失)       : {eval_results['eval_loss']:.4f}")
print(f"Accuracy (正解率) : {eval_results['eval_accuracy']:.4f} ({eval_results['eval_accuracy']*100:.2f}%)")

テキストをトークン化しています...
カスタムモデル (Max Pooling Architecture) を初期化しています...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


学習を開始します...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.279085,0.947550,0.509174
2,1.225492,0.947207,0.509174
3,1.137809,0.946526,0.509174



=== カスタムモデルによる最終評価 (Validation Set) ===


Loss (損失)       : 0.9465
Accuracy (正解率) : 0.5092 (50.92%)
